# 🥳 The Great Party Merge: Troubleshooting Guide - SOLUTIONS

Merging data in the real world is rarely as simple as a textbook example. In this notebook, we’ll use Ally and Bert’s guest lists to explore some common issues that may arise when data is messy, inconsistent, or conflicting. We will often create these issues ourselves to try out and see what happens when we encounter them. Go through all issues and take some time to understand what is going on here, what exactly causes the issue, and what consequences it might have.

For yourself: Try to annotate the notebook in plain English (jargon is not useful here!)

### NOTE: If the exercise is a Bonus ('BONUS:'), you can skip it. Only do it if you have time/want to try it out.

### 🛠️ The Setup
First, we will load some basic DataFrames to play around with. Check them thoroughly, note the differences in guest names and RSVP statuses between the two lists.


In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load Ally's List (that is the file: 'guestlist_ally.csv'); please keep the name 'df_a' for the dataframe.
df_a = pd.read_csv('guestlist_ally.csv') # csv file; only loads if in the same folder as your notebook!

# Load Bert's List (that is the file: 'guestlist_bert.pkl'); please keep the name 'df_b' for the dataframe.
df_b = pd.read_pickle('guestlist_bert.pkl') # pickle file

# Check what the dataframes look like in comparison
print("Ally's List:")
print(df_a.head(10)) # note length of the dataframe
print("\nBert's List:") # the \n makes sure there is an empty line in the output. Makes it look cleaner :) Also, python only prints the last line of code on the screen.
print(df_b.head(10)) # note length of the dataframe

Ally's List:
  GuestName RSVP_A
0     Alice    Yes
1       Bob     No
2   Charlie    Yes
3    Denise  Maybe
4      Lola    Yes
5       Gus  Maybe
6   Frankie    Yes
7      Gene     No
8    Corrie  Maybe

Bert's List:
  GuestName RSVP_B
0       Bob    Yes
1   Charlie    Yes
2     Elena     No
3     Frank    Yes
4    Denise  Maybe
5       Rob  Maybe


## 🔢 Issue 1: The Type Trap (Strings vs. Integers)
The Problem: If Ally uses a Guest_ID as a string ("101") and Bert uses an integer (101), pandas will treat them as entirely different keys and nothing will match.

Tasks:
1. Create the problem: IDs of different types in the two Guest Lists
2. Before you try inner-merging on 'Guest_ID', what do you think will happen and why? Type it in the Markdown cell below.
3. What does the (type of) error message mean? 
4. Fix it by using .astype(int) to convert Ally's IDs to match Bert's before merging.
5. BONUS: What would you do to make Bert's IDs match Ally's?
6. BONUS: Think about what object type works best for IDs. What sort of info is stored there? Is it always numeric or could you also expect other types of data in there?


### What will happen and why? Take an educated guess 🙂

Double click to enter your answer.

In [3]:
# 1. Create the problem
df_a_IDstring = df_a.copy() # this creates a copy df_a so we can experiment with it.
df_a_IDstring['Guest_ID'] = ["1", "2", "3", "4", "5", "6", "7", "8", "9"] # this adds a new ID variable in string format to experiment with different types

df_b['Guest_ID'] = [2, 3, 10, 11, 4, 12] # # this adds a new ID variable in integer format to experiment with different types. Note: These integers already take the structure of db_b and its overlap with df_a into account!

# Try inner-merge df_a_IDstring and df_b on 'Guest_ID':
df_final1 = pd.merge(df_a_IDstring, df_b, on="Guest_ID", how="inner")

ValueError: You are trying to merge on object and int64 columns for key 'Guest_ID'. If you wish to proceed you should use pd.concat

In [4]:
# How can you fix this for this specific merge (i.e., df_a_IDstring & df_b)? 
df_a_IDstring['Guest_ID'] = df_a_IDstring['Guest_ID'].astype(int) # recode the column 'GuestID' to integer
df_final1 = pd.merge(df_a_IDstring, df_b, on="Guest_ID", how="inner") # merge again

# Check what your final dataframe looks like - print (some of) it on the screen.
df_final1.head() # you see that the GuestName variable is now in there twice because it was not specified as merge key. Instead, custom suffixes were added to indicate from which dataframe the variable comes.

,GuestName_x,RSVP_A,Guest_ID,GuestName_y,RSVP_B
0,Bob,No,2,Bob,Yes
1,Charlie,Yes,3,Charlie,Yes
2,Denise,Maybe,4,Denise,Maybe


## 🍎🍐 Issue 2: Merging Apples and Pears (Merge keys do not have the same name)

The Problem: Ally and Bert did not talk enough to each other when they created their guest lists. So it seems that the variables on which the two lists should be merged have slightly different names...

Tasks: 
1. Create the problem: the merge key is not named in the same way in the two Guest Lists
2. Before you try inner-merging on 'Guest_ID', what do you think will happen and why? Type it in the Markdown cell below.
3. What does the (type of) error message mean?
4. Try out the two solutions and check the questions in the last cell (# 3. Inspect Solutions).

### What will happen and why? Take an educated guess 🙂

Double click to enter your answer.

In [5]:
# 1. Create the problem
df_b_diff = df_b.copy() # this creates a copy for experimenting
df_b_diff = df_b_diff.rename(columns={"Guest_ID": "GuestID"}) # small difference, large consequence: 

# Add the Guest_ID to df_a (because we only added it to df_a_IDstring above for Issue 1!)
df_a['Guest_ID'] = [1, 2, 3, 4, 5, 6, 7, 8, 9]

# Inspect the change in df_b and df_b_diff
# Check what the dataframes look like in comparison
print("df_b")
print(df_b.head(10)) # note length of the dataframe
print("\ndf_b_diff:") # the \n makes sure there is an empty line in the output. Makes it look cleaner :) Also, python only prints the last line of code on the screen.
print(df_b_diff.head(10)) # note length of the dataframe

df_b
  GuestName RSVP_B  Guest_ID
0       Bob    Yes         2
1   Charlie    Yes         3
2     Elena     No        10
3     Frank    Yes        11
4    Denise  Maybe         4
5       Rob  Maybe        12

df_b_diff:
  GuestName RSVP_B  GuestID
0       Bob    Yes        2
1   Charlie    Yes        3
2     Elena     No       10
3     Frank    Yes       11
4    Denise  Maybe        4
5       Rob  Maybe       12


In [6]:
# 2. Inner merge on 'Guest_ID':
df_final2 = pd.merge(df_a, df_b_diff, on="Guest_ID", how="inner")

KeyError: 'Guest_ID'

In [7]:
# 3. Solution 1: specifying both keys explicitly (--> faster)
df_final2_s1 = pd.merge(df_a, df_b_diff, left_on="Guest_ID", right_on="GuestID", how="inner") # left dataframe is the first one listed (since we are reading from left to right!)

In [8]:
# 3. Solution 2: renaming column in one dataframe to match the other (--> cleaner)
df_b_diff = df_b_diff.rename(columns={"GuestID": "Guest_ID"})
df_final2_s2 = pd.merge(df_a, df_b_diff, on="Guest_ID", how="inner")

In [9]:
# 3. Inspect Solutions
print("Solution 1:")
print(df_final2_s1.head(10)) 
print("\nSolution 2") # the \n makes sure there is an empty line in the output. Makes it look cleaner :) Also, python only prints the last line of code on the screen.
print(df_final2_s2.head(10)) 

# How many observations are stored in this dataframe and why? 
# Some column names have the suffixes _x and _y? Why? 
# What happens to merge keys in Solution 1 vs. Solution 2?

Solution 1:
  GuestName_x RSVP_A  Guest_ID GuestName_y RSVP_B  GuestID
0         Bob     No         2         Bob    Yes        2
1     Charlie    Yes         3     Charlie    Yes        3
2      Denise  Maybe         4      Denise  Maybe        4

Solution 2
  GuestName_x RSVP_A  Guest_ID GuestName_y RSVP_B
0         Bob     No         2         Bob    Yes
1     Charlie    Yes         3     Charlie    Yes
2      Denise  Maybe         4      Denise  Maybe


## 🧐 Issue 3: The Hidden Match

The Problem: Computers are literal. To Python, "Alice" and "alice" are two different people. If Ally uses Capital Case and Bert uses lowercase, your merge will fail to find matches.

Tasks: 
1. Create a version of Bert's list where all names in the GuestName column are lowercase.
2. Before you try a standard inner merge on GuestName, what do you think will happen and why? Type it in the Markdown cell below!
3. What does the (type of) error message mean?
4. Try to fix it by, for example, using .str.lower() to normalize names in df_a before merging.

### What will happen and why? Take an educated guess 🙂

Double click to enter your answer.

In [10]:
# 1. Create the problem
df_b_low = df_b.copy() # create a copy of df_b to experiment with - leaves df_b untouched.
df_b_low['GuestName'] = df_b_low['GuestName'].str.lower() # convert all strings in variable 'GuestName' to lower case

# Inspect df_b and df_b_low to see what changed
print("df_b:")
print(df_b.head(10)) 
print("\ndf_b_low")
print(df_b_low.head(10)) 

df_b:
  GuestName RSVP_B  Guest_ID
0       Bob    Yes         2
1   Charlie    Yes         3
2     Elena     No        10
3     Frank    Yes        11
4    Denise  Maybe         4
5       Rob  Maybe        12

df_b_low
  GuestName RSVP_B  Guest_ID
0       bob    Yes         2
1   charlie    Yes         3
2     elena     No        10
3     frank    Yes        11
4    denise  Maybe         4
5       rob  Maybe        12


In [11]:
# 2. Try an inner merge of df_a and df_b_low
df_final3 = pd.merge(df_a, df_b_low, on="GuestName", how="inner")
df_final3.head() # Ok, this looks very strange!

,GuestName,RSVP_A,Guest_ID_x,RSVP_B,Guest_ID_y


In [12]:
# 3. How can you fix this for this specific merge (i.e., df_a and df_b_low)? 
df_a['GuestName'] = df_a['GuestName'].str.lower() # converts all strings in df_a to lower case to allow matching
df_final3 = pd.merge(df_a, df_b_low, on = 'GuestName', how = 'inner') # merge again
df_final3.head()

,GuestName,RSVP_A,Guest_ID_x,RSVP_B,Guest_ID_y
0,bob,No,2,Yes,2
1,charlie,Yes,3,Yes,3
2,denise,Maybe,4,Maybe,4


## 👯 Issue 4: The Double-Entry (Duplicates)

The Problem: Ally was a little under the weather and forgot that she had entered Bob already - she now enters him again, just to be sure! What happens if "Bob" is listed twice in Ally's list? A standard merge creates a so-called Cartesian Product (creates rows for ALL possible matchings). If Bob is in List A twice and List B once, you get two Bobs. If he's in both twice, you get four Bobs!

Tasks:
1. Create the problem: a duplicate Bob in Ally's list
2. Before you inner-merge df_a_dupes with df_b on 'GuestName', what do you think will happen and why? Type it in the Markdown cell below!
3. What does the (type of) error message mean?
4. Fix it by using .drop_duplicates() on df_a_dupes before merging.
5. BONUS: Think about when duplicates could be expected (what if Ally really has two friends named Bob?). How could you, in theory, solve that? 
6. BONUS: Create a duplicate Bob in Bert's list as well and try again. What happens? Why?

### What will happen and why? Take an educated guess 🙂

Double click to enter your answer.

In [15]:
# 1. Create the problem: add another bob to Ally's guest list and one to Bert's guest list
df_a_dupes = pd.concat([df_a, pd.DataFrame({"GuestName": ["bob"], "RSVP_A": ["No"], "Guest_ID": ["2"]})], ignore_index=True) # without the ignore_index = True, python will assign the index 0 to this new row!
# note: there was actually a mistake in the notebook we use in class: for the concatenation, we forgot to specify the Guest_ID variable for the duplicate Bob.

df_b['GuestName'] = df_b['GuestName'].str.lower() #convert string in df_b into lower case, otherwise we will have the same issue as in Isse 3!

In [16]:
# 2. Try inner-merge df_a_dupes with df_b_dupes on 'GuestName' and check what happens
df_final4 = pd.merge(df_a_dupes, df_b, on="GuestName", how="inner")
df_final4 # I see bob 2 times in there. But for it was there only once in the other dataframe...

,GuestName,RSVP_A,Guest_ID_x,RSVP_B,Guest_ID_y
0,bob,No,2,Yes,2
1,charlie,Yes,3,Yes,3
2,denise,Maybe,4,Maybe,4
3,bob,No,2,Yes,2


In [17]:
# 3. Fix
df_a_dupes = df_a_dupes.drop_duplicates(['GuestName']) # removes the duplicate Bob that we had added to create this problem.
df_a_dupes

,GuestName,RSVP_A,Guest_ID
0,alice,Yes,1
1,bob,No,2
2,charlie,Yes,3
3,denise,Maybe,4
4,lola,Yes,5
5,gus,Maybe,6
6,frankie,Yes,7
7,gene,No,8
8,corrie,Maybe,9


In [18]:
# 4. Merge again and inspect
df_final4 = pd.merge(df_a_dupes, df_b, on="GuestName", how="inner")
df_final4 # gives us three rows instead of 4 :)

,GuestName,RSVP_A,Guest_ID_x,RSVP_B,Guest_ID_y
0,bob,No,2,Yes,2
1,charlie,Yes,3,Yes,3
2,denise,Maybe,4,Maybe,4


## Tips 

In the following, two tips are discussed. How could they be helpful for you?

### 1. 🕵️ Using Indicator

Sometimes you merge the lists and the result is different from what you expected. You need to know why. Was the guest only on Ally's list? Only on Bert's? The indicator=True argument adds a _merge column showing the source of the row. Let the code below run and check what kind of info the indicator gives you.


In [19]:
# Outer merge using indicator
merged_audit = pd.merge(df_a, df_b, on="GuestName", how="outer", indicator=True) # creates a variable '_merge'

print(merged_audit['_merge'].value_counts()) # print the _merge variable on the screen

_merge
left_only     6
right_only    3
both          3
Name: count, dtype: int64


### 🛠️ 2. Custom Suffixes
By default, pandas adds _x and _y when variables in two merged dataframes have the same name (not the variable on which is merged). You can make your code much more readable by using the suffixes argument. Let the code below run and see what it gives you.

In [22]:
# Create the problem: variable names with same name
df_a_collision = df_a.rename(columns={'RSVP_A': 'RSVP'}) # note that our own variables already have different names :) so this would not be a problem here anyway.
df_b_collision = df_b_low.rename(columns={'RSVP_B': 'RSVP'}) # we therefore rename them to have the same name, thus creating our problem.

# Inspect df_b and df_b_low to see what changed
print("df_a_collision:")
print(df_a_collision.head(10)) 
print("\ndf_b_collision")
print(df_b_collision.head(10)) 

df_a_collision:
  GuestName   RSVP  Guest_ID
0     alice    Yes         1
1       bob     No         2
2   charlie    Yes         3
3    denise  Maybe         4
4      lola    Yes         5
5       gus  Maybe         6
6   frankie    Yes         7
7      gene     No         8
8    corrie  Maybe         9

df_b_collision
  GuestName   RSVP  Guest_ID
0       bob    Yes         2
1   charlie    Yes         3
2     elena     No        10
3     frank    Yes        11
4    denise  Maybe         4
5       rob  Maybe        12


In [23]:
# Add custom suffixes to be able to distinguish which variables came from which dataframe
df_clean = pd.merge(df_a_collision, df_b_collision, on="GuestName", how="outer", suffixes=('_Ally', '_Bert'))
df_clean.head(15)

,GuestName,RSVP_Ally,Guest_ID_Ally,RSVP_Bert,Guest_ID_Bert
0,alice,Yes,1.0,NaN,NaN
1,bob,No,2.0,Yes,2.0
2,charlie,Yes,3.0,Yes,3.0
3,corrie,Maybe,9.0,NaN,NaN
4,denise,Maybe,4.0,Maybe,4.0
5,elena,NaN,NaN,No,10.0
6,frank,NaN,NaN,Yes,11.0
7,frankie,Yes,7.0,NaN,NaN
8,gene,No,8.0,NaN,NaN
9,gus,Maybe,6.0,NaN,NaN


## ⚔️ Bonus Issue to check out: Conflicting RSVPs

The Problem: Look at Bob. Ally marked him as "No," but Bert marked him as "Yes." When you merge, pandas cannot match RSVPs directly but creates two columns for storing conflicting RSVPs for the same guest.

What is done here?

- First, the variables RSVP_A and RSVP_B are renamed to have the same name (creating the problem).
- We then use the suffixes=('_Ally', '_Bert') argument in our merge to make the columns clear. So in the merged version we can see clearly where variables come from and where there is a conflict in values that python cannot resolve on its own.

In [24]:
# Create the problem: Two new dataframes with variables of same names
df_a_conflict = df_a
df_a_conflict = df_a_conflict.rename(columns={"RSVP_A": "RSVP"}) # Remember: RSVP is not the merge variable!

df_b_conflict = df_b_low
df_b_conflict = df_b_conflict.rename(columns={"RSVP_B": "RSVP"}) # Remember: RSVP is not the merge variable!

# Merge dataframes with custom suffixes that help you understand which data comes from which dataframe
df_conflict = pd.merge(df_a_conflict, df_b_conflict, on="GuestName", how="inner", suffixes=('_Ally', '_Bert'))
df_conflict.head(10)

,GuestName,RSVP_Ally,Guest_ID_Ally,RSVP_Bert,Guest_ID_Bert
0,bob,No,2,Yes,2
1,charlie,Yes,3,Yes,3
2,denise,Maybe,4,Maybe,4


### Bonus Issue continued:

- We create a Final_RSVP column in which we resolve the conflict. We specify the following conditions: If Ally's list says "Maybe" but Bert's says "Yes," we'll take Bert's "Yes." Otherwise, we trust Ally. Note: These specifications depend entirely on how you think this should be solved!
- We print the final dataframe to check if it worked and what it looks like.

Inspect the result: What do you see? What happened?

In [ ]:
# Now we use np.where (from numpy) to resolve the conflict
df_conflict['Final_RSVP'] = np.where( # first, create a new column for what you are about to check; then use 'np.where' which is like 'ifelse', but does not need a loop for iterating over all rows of the df
    (df_conflict['RSVP_Ally'] == "Maybe") & (df_conflict['RSVP_Bert'] == "Yes"), # specify condition
    df_conflict['RSVP_Bert'], # 1st step (specified by the people who wrote the np.where function), meaning: put this in if the condition is met (== TRUE)
    df_conflict['RSVP_Ally'] # 2nd step: put this in if the condition is not met (== FALSE)
)

df_conflict[['GuestName', 'RSVP_Ally', 'RSVP_Bert', 'Final_RSVP']] # show the final dataframe with only relevant columns displayed